In [25]:
import fasttext

In [26]:
model = fasttext.load_model("sql_injection_detection.ftz")

In [27]:
import re

In [28]:


def preprocess_data(query: str) -> str:
    query = query.lower()

    query = re.sub(r"--", " -- ", query)
    query = re.sub(r"/\*", " /* ", query)
    query = re.sub(r"\*/", " */ ", query)

    query = re.sub(r"'(?:''|[^'])*'", " VAL ", query)

    query = re.sub(r"\bnull\b", " VAL ", query)

    query = re.sub(r"\b\d+(\.\d+)?([eE][+-]?\d+)?\b", " VAL ", query)

    query = re.sub(r"([=<>!]+)", r" \1 ", query)
    query = re.sub(r"([(),;])", r" \1 ", query)

    query = re.sub(r"\s+", " ", query).strip()

    return query

In [29]:
import time

In [30]:
import numpy as np

In [7]:
sql_query = """create table #SVer(ID int, Name sysname, Internal_Value int, Value nvarchar(512)) insert #SVer exec master.dbo.xp_msver insert #SVer select t.* from sys.dm_os_host_info CROSS APPLY ( VALUES (1001, 'host_platform', 0, host_platform), (1002, 'host_distribution', 0, host_distribution), (1003, 'host_release', 0, host_release), (1004, 'host_service_pack_level', 0, host_service_pack_level), (1005, 'host_sku', host_sku, ''), (1006, 'HardwareGeneration', '', ''), (1007, 'ServiceTier', '', ''), (1008, 'ReservedStorageSizeMB', '0', '0'), (1009, 'UsedStorageSizeMB', '0', '0') ) t(id, [name], internal_value, [value]) -- Managed Instance-specific properties if (SERVERPROPERTY('EngineEdition') = 8) begin DECLARE @gen4memoryPerCoreMB float = 7168.0 DECLARE @gen5memoryPerCoreMB float = 5223.0 DECLARE @physicalMemory float DECLARE @virtual_core_count int DECLARE @reservedStorageSize bigint DECLARE @usedStorageSize decimal(18,2) DECLARE @hwGeneration nvarchar(128) DECLARE @serviceTier nvarchar(128) SET @physicalMemory = (SELECT TOP 1 [virtual_core_count] * ( CASE WHEN [hardware_generation] = 'Gen4' THEN @gen4memoryPerCoreMB WHEN [hardware_generation] = 'Gen5' THEN @gen5memoryPerCoreMB ELSE 0 END ) FROM master.sys.server_resource_stats ORDER BY start_time DESC) IF (@physicalMemory <> 0) BEGIN UPDATE #SVer SET [Internal_Value] = @physicalMemory WHERE Name = N'PhysicalMemory' UPDATE #SVer SET [Value] = CONCAT( @physicalMemory, ' (', @physicalMemory * 1024, ')') WHERE Name = N'PhysicalMemory' END UPDATE #SVer SET [Internal_Value] = (SELECT TOP 1 [virtual_core_count] FROM master.sys.server_resource_stats ORDER BY start_time desc) WHERE Name = N'ProcessorCount' UPDATE #SVer SET [Value] = [Internal_Value] WHERE Name = N'ProcessorCount' SELECT TOP 1 @hwGeneration = [hardware_generation], @serviceTier =[sku], @virtual_core_count = [virtual_core_count], @reservedStorageSize = [reserved_storage_mb], @usedStorageSize = [storage_space_used_mb] FROM master.sys.server_resource_stats ORDER BY [start_time] DESC UPDATE #SVer SET [Value] = @hwGeneration WHERE Name = N'HardwareGeneration' UPDATE #SVer SET [Value] = @serviceTier WHERE Name = N'ServiceTier' UPDATE #SVer SET [Value] = @reservedStorageSize WHERE Name = N'ReservedStorageSizeMB' UPDATE #SVer SET [Value] = @usedStorageSize WHERE Name = N'UsedStorageSizeMB' end SELECT CAST( serverproperty(N'Servername') AS sysname) AS [Server_Name], 'Server[@Name=' + quotename(CAST( serverproperty(N'Servername') AS sysname),'''') + ']' AS [Server_Urn], CAST(null AS int) AS [Server_ServerType], 0 AS [Server_IsContainedAuthentication], (@@microsoftversion / 0x1000000) & 0xff AS [VersionMajor], (@@microsoftversion / 0x10000) & 0xff AS [VersionMinor], @@microsoftversion & 0xffff AS [BuildNumber], CAST(SERVERPROPERTY('IsSingleUser') AS bit) AS [IsSingleUser], CAST(SERVERPROPERTY(N'Edition') AS sysname) AS [Edition], CAST(SERVERPROPERTY('EngineEdition') AS int) AS [EngineEdition], CAST(ISNULL(SERVERPROPERTY(N'IsXTPSupported'), 0) AS bit) AS [IsXTPSupported], SERVERPROPERTY(N'ProductVersion') AS [VersionString], ( select Value from #SVer where Name =N'host_platform') AS [HostPlatform], CAST(FULLTEXTSERVICEPROPERTY('IsFullTextInstalled') AS bit) AS [IsFullTextInstalled] ORDER BY [Server_Name] ASC drop table #SVer"""

In [14]:
sql_query = "DECLARE @edition sysname; SET @edition = cast(SERVERPROPERTY(N'EDITION') as sysname); SELECT case when @edition = N'SQL Azure' then 2 else 1 end as 'DatabaseEngineType', SERVERPROPERTY('EngineEdition') AS DatabaseEngineEdition, SERVERPROPERTY('ProductVersion') AS ProductVersion, @@MICROSOFTVERSION AS MicrosoftVersion; select host_platform from sys.dm_os_host_info if @edition = N'SQL Azure' select 'TCP' as ConnectionProtocol else exec ('select CONVERT(nvarchar(40),CONNECTIONPROPERTY(''net_transport'')) as ConnectionProtocol')"

In [57]:
sql_query = """
select 
  SERVERPROPERTY;
if serverproperty = 5 begin exec sp_executesql end else begin 
select 
  '' end if serverproperty not in (6) begin begin try exec sp_executesql  end try begin catch 
"""

In [58]:
preprocess_data(sql_query)

'select serverproperty ; if serverproperty = VAL begin exec sp_executesql end else begin select VAL end if serverproperty not in ( VAL ) begin begin try exec sp_executesql end try begin catch'

In [59]:

start = time.perf_counter()
label, accuracy = model.predict(preprocess_data(sql_query))
end = time.perf_counter()
print(label[0])
print(accuracy)
print(f"Prediction took: {(end - start) * 1000:.4f} ms")

__label__1
[0.92295092]
Prediction took: 1.2712 ms
